# How to insert data into OLMo training

In [ ]:
from olmo.data import build_train_dataloader

from olmo.config import TrainConfig
from olmo.tokenizer import Tokenizer

from olmo_data_insertion import create_olmo_insert_dict
import os
from pathlib import Path

## Create a file with the insertions

In [ ]:
olmo_config_path = "../configs/official-0425/OLMo2-1B-stage1.yaml"
global_indices_path = "global_indices.npy"

# this is how we specify the text that should be inserted. Global token position: Text
insert_dict = {5: "This text will be inserted at token position 5", 
               4096: "Another inserted text", 
               2*4096-2: "This text goes over 2 different sequences, so the insert position will be corrected."}

# this function transforms the intuitive insert_dict into another data structure that makes it easy to insert the text into olmos training data later
memmap_insert_dict = create_olmo_insert_dict(insert_dict, olmo_config_path, global_indices_path=global_indices_path)

print(memmap_insert_dict)

In [ ]:
# save the memmap_insert_dict to file
import pickle
with open("insert_dict.pkl", "wb") as f:
    pickle.dump(memmap_insert_dict, f)

In [ ]:
# to tell the olmo training script to use the memmap_insert_dict, we specify an environment variable
os.environ['OLMO_EXPERIMENT_INSERTIONS_FILE'] =  '/Users/sbordt/Nextcloud/OLMo/training-data-insertion/insert_dict.pkl'

## Now, go through the olmo data loading pipeline to see that our data is added 

In [ ]:
# check environment variables
print("Environment variables:")
for key, value in os.environ.items():
    if key.startswith('OLMO_'):
        print(f"{key}: {value}")

file_path = os.getenv('OLMO_EXPERIMENT_INSERTIONS_FILE', None)
if file_path and Path(file_path).exists():
    print(f"File {file_path} exists.")
else:
    print(f"File {file_path} does not exist or is not set.")

In [ ]:
cfg = TrainConfig.load(olmo_config_path)
sequence_length = cfg.model.max_sequence_length
tokenizer = Tokenizer.from_train_config(cfg)
cfg.device_train_batch_size = 2 # if we do not set this we get an assertion error in build_train_dataloader
cfg.save_overwrite = True # if we do not set this, we get an error if the folder already exists. might want to change this in the future.
dataloader = build_train_dataloader(cfg)
dataset = dataloader.dataset

In [ ]:
type(dataset.dataset) # this should be PatchedMemmapDataset

In [ ]:
# load the first training batch
batch = next(iter(dataloader))

In [ ]:
# print the first sequence in the first training batch
print(tokenizer.decode(batch['input_ids'][0].cpu().numpy(), skip_special_tokens=False))

In [ ]:
# print the second sequence in the first training batch
print(tokenizer.decode(batch['input_ids'][1].cpu().numpy(), skip_special_tokens=False))